# Reasoning Models & Test-Time Compute — Hands-On

**LLM Engineering · Domain 2 · Roadmap Weeks 11/16**

Companion to `02 Literature Notes/LLM Engineering/Reasoning Models`. Runs offline by simulating a noisy reasoner.

## 0. Setup — a noisy 'reasoner' correct 55% of the time

In [ ]:
%pip install -q numpy
import random, numpy as np
from collections import Counter
random.seed(0)
CORRECT = 42
def reason_once(prompt, p_correct=0.55):
    return "steps...", (CORRECT if random.random() < p_correct else random.choice([40,41,43,44]))
print("ok")

## 1. Self-consistency: more chains -> higher accuracy

In [ ]:
def self_consistency(n, p=0.55, trials=500):
    wins = 0
    for _ in range(trials):
        answers = [reason_once("q", p)[1] for _ in range(n)]
        if Counter(answers).most_common(1)[0][0] == CORRECT: wins += 1
    return wins / trials
for n in (1, 3, 5, 11, 21):
    print(f"n={n:>2} chains  accuracy={self_consistency(n):.2%}")
print("majority voting turns a 55% one-shot into much higher accuracy")

> This is compute-for-accuracy: N× tokens buys a real accuracy gain — but with diminishing returns.

## 2. Diminishing returns and cost

In [ ]:
base = self_consistency(1)
for n in (1, 5, 25, 51):
    acc = self_consistency(n)
    print(f"n={n:>2}  acc={acc:.2%}  cost~{n:>2}x  acc-per-cost={ (acc-base)/n:.4f}")

## 3. Self-consistency needs diversity (temperature>0)

In [ ]:
# if all chains are identical (temp=0), voting adds nothing
def no_diversity(n):
    fixed = reason_once("q")[1]         # one deterministic chain, repeated
    answers = [fixed]*n
    return Counter(answers).most_common(1)[0][0]
print("temp=0 (identical chains) majority:", no_diversity(21), "-> no benefit from N")

## 4. Match compute to difficulty

In [ ]:
def difficulty(q):
    return 0.9 if any(w in q.lower() for w in ("prove","why","derive","plan")) else 0.2
def route(q, threshold=0.5):
    return ("reasoning", 10) if difficulty(q) >= threshold else ("cheap-1shot", 1)
queries = ["Capital of France?","Prove sqrt(2) irrational","2+2?","Plan a migration"]
total = 0
for q in queries:
    mode, cost = route(q); total += cost
    print(f"[{mode:>11} ~{cost:>2}x] {q}")
print("total cost:", total, "vs", 10*len(queries), "if everything used reasoning")

## 5. Exercises
1. Plot accuracy vs n for p_correct in {0.4, 0.55, 0.7}. When does voting fail (<0.5)?
2. Add a verifier that rejects impossible answers before voting.
3. Tune the difficulty threshold to hit a target average cost.
4. Simulate tree-of-thoughts: expand 2 branches, keep the higher-scored one.

## Links
- Literature note: `02 Literature Notes/LLM Engineering/Reasoning Models`
- Snippets: `04 Code Snippets/LLM/Self-Consistency Majority Vote`, `.../Match Test-Time Compute to Difficulty`
- MOC: `06 Maps of Content/LLM Engineering Concepts`